# Supplementary Tutorial: The Proprioceptive Noise Chain
**Computational Sensorimotor Control — Week 9**

**Purpose:** Derive how spindle noise maps through the moment arm matrix and Jacobian
to produce anisotropic uncertainty ellipses at the hand — for both position and velocity.

**Structure:**
- Parts 1–4: Static derivation and visualization (foundation)
- Parts 5–6: Configuration dependence and sampling
- **Part 7: Position chain l(t) during a 10 cm reach**
- **Part 8: Velocity chain dl/dt(t) during the same reach**
- **Part 9: Both channels together during the reach**

All code is provided — run each cell in order.


In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Patch
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3, 'font.family': 'serif'})
from smc import Arm, Q_REF, L1, L2, make_muscles, Sensor

arm = Arm()
fk = arm.forward_kinematics
def ik(p): return arm.inverse_kinematics(p[0], p[1])
Jfn = arm.jacobian
muscles = make_muscles()
dt = 0.001

NAVY='#1B2A4A'; TEAL='#2E86AB'; RED='#E74C3C'
GREEN='#27AE60'; GRAY='#7F8C8D'; ORANGE='#E67E22'; PURPLE='#8E44AD'

def draw_arm(ax, q, color=NAVY, lw=4, alpha=0.6, label=None):
    shoulder = np.array([0.0, 0.0])
    elbow = shoulder + L1*np.array([np.cos(q[0]), np.sin(q[0])])
    hand = elbow + L2*np.array([np.cos(q[0]+q[1]), np.sin(q[0]+q[1])])
    ax.plot([shoulder[0]*100,elbow[0]*100],[shoulder[1]*100,elbow[1]*100],
            color=color,lw=lw,alpha=alpha,solid_capstyle='round',label=label)
    ax.plot([elbow[0]*100,hand[0]*100],[elbow[1]*100,hand[1]*100],
            color=color,lw=lw,alpha=alpha,solid_capstyle='round')
    ax.plot(*shoulder*100,'o',color=color,ms=10,alpha=alpha,zorder=5)
    ax.plot(*elbow*100,'o',color=color,ms=7,alpha=alpha,zorder=5)
    ax.plot(*hand*100,'o',color=color,ms=5,alpha=min(1,alpha+0.2),zorder=5)
    return shoulder*100, elbow*100, hand*100

def cov_ellipse(ax, mu, cov, n_std=2, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w = 2*n_std*np.sqrt(max(vals[1],0))
    h = 2*n_std*np.sqrt(max(vals[0],0))
    e = Ellipse(mu, w, h, angle=angle, **kwargs)
    ax.add_patch(e); return e

print(f'Arm: L1={L1:.2f}m, L2={L2:.2f}m')
print(f'Q_REF = ({Q_REF[0]*180/np.pi:.1f}°, {Q_REF[1]*180/np.pi:.1f}°)')
print(f'Muscles: {[m.name for m in muscles]}')

---
## Part 1: Spindle Measurements

Group II afferents encode muscle length $l(q)$. Group Ia afferents encode velocity $dl/dt$ — directly, not by differentiating.

In [ ]:
q = np.array(Q_REF); qd = np.zeros(2)
print(f'{"Muscle":>8s}  {"Length (cm)":>12s}  {"Velocity":>14s}  {"r_sh":>6s}  {"r_el":>6s}')
print('-'*60)
for m in muscles:
    print(f'{m.name:>8s}  {m.length(q)*100:12.2f}  {m.velocity(qd):14.4f}  {m.r_sh:6.3f}  {m.r_el:6.3f}')

---
## Part 2: Moment Arm Mapping (Muscles → Joints)

The A matrix (6×2) maps joint changes to muscle changes. Both position and velocity chains use the **same** $(A^T A)^{-1}$.

In [ ]:
A = np.array([[m.r_sh, m.r_el] for m in muscles])
ATA = A.T @ A
ATA_inv = np.linalg.inv(ATA)
A_pinv = ATA_inv @ A.T

print(f'(A^T A)^{{-1}} diagonal: [{ATA_inv[0,0]:.2f}, {ATA_inv[1,1]:.2f}]')
print(f'Ratio: {np.sqrt(ATA_inv[1,1]/ATA_inv[0,0]):.2f} \u2014 elbow noisier than shoulder')

# Calibrate sigma_l so full chain gives ~12mm mean hand SD
J_ref = Jfn(Q_REF)
s_test = 0.001
R_test = J_ref @ (s_test**2 * ATA_inv) @ J_ref.T
sigma_l = s_test * (0.012 / np.sqrt(np.mean(np.diag(R_test))))
sigma_v = 0.005  # 5 mm/s Ia noise

R_q = sigma_l**2 * ATA_inv
R_qd = sigma_v**2 * ATA_inv

print(f'\nCalibrated \u03c3_l = {sigma_l*1000:.2f} mm')
print(f'\u03c3_v = {sigma_v*1000:.0f} mm/s')
print(f'Joint pos noise: \u03c3_q1={np.sqrt(R_q[0,0])*180/np.pi:.2f}\u00b0, \u03c3_q2={np.sqrt(R_q[1,1])*180/np.pi:.2f}\u00b0')
print(f'Joint vel noise: \u03c3_qd1={np.sqrt(R_qd[0,0])*180/np.pi:.2f}\u00b0/s, \u03c3_qd2={np.sqrt(R_qd[1,1])*180/np.pi:.2f}\u00b0/s')

---
## Part 3: The Jacobian (Joints → Hand)

$R_{hand} = J R_q J^T$ and $R_{hand,vel} = J R_{\dot{q}} J^T$. Both use the same $J$ and $(A^TA)^{-1}$ — same ellipse shape, different scale.


In [ ]:
J = Jfn(Q_REF)
R_hand = J @ R_q @ J.T
R_hand_vel = J @ R_qd @ J.T
eigvals_p, eigvecs_p = np.linalg.eigh(R_hand)
eigvals_v, _ = np.linalg.eigh(R_hand_vel)

print('Position chain:')
print(f'  \u03c3_x={np.sqrt(R_hand[0,0])*1000:.1f}mm, \u03c3_y={np.sqrt(R_hand[1,1])*1000:.1f}mm, mean={np.sqrt(np.mean(np.diag(R_hand)))*1000:.1f}mm')
print(f'  ratio = {np.sqrt(eigvals_p[1]/eigvals_p[0]):.1f}:1')
print('Velocity chain:')
print(f'  \u03c3_vx={np.sqrt(R_hand_vel[0,0])*1000:.1f}mm/s, \u03c3_vy={np.sqrt(R_hand_vel[1,1])*1000:.1f}mm/s, mean={np.sqrt(np.mean(np.diag(R_hand_vel)))*1000:.1f}mm/s')
print(f'  ratio = {np.sqrt(eigvals_v[1]/eigvals_v[0]):.1f}:1')
print(f'\nSame ratio: {np.sqrt(eigvals_p[1]/eigvals_p[0]):.1f}:1 = {np.sqrt(eigvals_v[1]/eigvals_v[0]):.1f}:1 \u2714')

---
## Part 4: Visualize Both Ellipses at Q_REF

Both chains produce ellipses with the **same orientation** because they share the same $J$ and $(A^TA)^{-1}$.
One is in position units (mm), the other in velocity units (mm/s) — we plot them in separate panels
with their own axes so both are clearly visible.

**Panel A (Position):** The purple dashed ellipse shows ±2σ proprioceptive position uncertainty
(from group II afferents, mapped through the full noise chain). The orange circle shows ±2σ
peripheral vision uncertainty (σ = 5 mm, isotropic) for comparison — this is a position-vs-position
comparison of the two hand-sensing channels at the same posture.

**Panel B (Velocity):** The purple dashed ellipse shows ±2σ proprioceptive velocity uncertainty
(from Ia afferents, mapped through the same chain). There is no visual velocity comparison
because we have not modeled visual motion processing (MT/V5).

**Panel C:** Both ellipses normalized to unit peak and overlaid — showing they have identical shape and
orientation because they share the same J and $(A^TA)^{-1}$.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))

# ── Panel A: Position ellipse ZOOMED to hand region ──
ax = axes[0]
hd = np.array(fk(Q_REF)) * 100  # hand in cm
R_hand_cm2 = R_hand * 100**2
cov_ellipse(ax, hd, R_hand_cm2, n_std=2,
            facecolor=PURPLE, alpha=0.15, edgecolor=PURPLE, lw=2, ls="--",
            label=r"Proprio $\pm 2\sigma$ (position, group II)")
cov_ellipse(ax, hd, np.eye(2)*0.5**2, n_std=2,
            facecolor=ORANGE, alpha=0.2, edgecolor=ORANGE, lw=2,
            label=r"Peripheral vision $\pm 2\sigma$ ($\sigma$=5mm, isotropic)")
# Eigenvector arrows
for i in range(2):
    d = eigvecs_p[:,i] * np.sqrt(eigvals_p[i]) * 100 * 4
    ax.annotate("", xy=hd+d, xytext=hd,
                arrowprops=dict(arrowstyle="->", color=PURPLE, lw=2))
ax.plot(*hd, "ko", ms=6, zorder=5)
ax.text(hd[0]+0.3, hd[1]-0.5, "Hand", fontsize=10)
ax.set_aspect("equal"); ax.legend(fontsize=8, loc="upper left")
ax.set_xlabel("x (cm)"); ax.set_ylabel("y (cm)")
ax.set_title(f"A: Position uncertainty at Q_REF\n"
             f"$\\sigma_{{major}}$={np.sqrt(eigvals_p[1])*1000:.1f}mm, "
             f"$\\sigma_{{minor}}$={np.sqrt(eigvals_p[0])*1000:.1f}mm, "
             f"ratio={np.sqrt(eigvals_p[1]/eigvals_p[0]):.1f}:1",
             fontweight="bold", fontsize=10)
ax.set_xlim(hd[0]-3, hd[0]+3); ax.set_ylim(hd[1]-3, hd[1]+3)

# ── Panel B: Velocity ellipse in its own units (mm/s) ──
ax = axes[1]
eigvals_v_plot, eigvecs_v_plot = np.linalg.eigh(R_hand_vel)
R_vel_mms2 = R_hand_vel * 1e6  # m²/s² → mm²/s²
cov_ellipse(ax, [0, 0], R_vel_mms2, n_std=2,
            facecolor=PURPLE, alpha=0.15, edgecolor=PURPLE, lw=2, ls="--",
            label=r"Proprio velocity $\pm 2\sigma$ (Ia afferents)")
for i in range(2):
    d = eigvecs_v_plot[:,i] * np.sqrt(eigvals_v_plot[i]) * 1000 * 4
    ax.annotate("", xy=d, xytext=[0,0],
                arrowprops=dict(arrowstyle="->", color=PURPLE, lw=2))
ax.axhline(0, color="black", lw=0.5, alpha=0.3)
ax.axvline(0, color="black", lw=0.5, alpha=0.3)
ax.set_aspect("equal"); ax.legend(fontsize=8)
ax.set_xlabel("vx noise (mm/s)"); ax.set_ylabel("vy noise (mm/s)")
ax.set_title(f"B: Velocity uncertainty at Q_REF\n"
             f"$\\sigma_{{major}}$={np.sqrt(eigvals_v_plot[1])*1000:.0f}mm/s, "
             f"$\\sigma_{{minor}}$={np.sqrt(eigvals_v_plot[0])*1000:.0f}mm/s, "
             f"ratio={np.sqrt(eigvals_v_plot[1]/eigvals_v_plot[0]):.1f}:1",
             fontweight="bold", fontsize=10)
ax.set_xlim(-120, 120); ax.set_ylim(-80, 80)

# ── Panel C: Overlay both to show same orientation ──
ax = axes[2]
# Normalize: scale each covariance so its major eigenvalue = 1
ev_p = np.linalg.eigh(R_hand)[0]
ev_v = np.linalg.eigh(R_hand_vel)[0]
R_pos_norm = R_hand / ev_p.max()
R_vel_norm = R_hand_vel / ev_v.max()
# Use distinct styles so both are visible even when overlapping
cov_ellipse(ax, [0, 0], R_pos_norm, n_std=2,
            facecolor="none", edgecolor=PURPLE, lw=3, ls="--",
            label="Position (normalized)")
cov_ellipse(ax, [0, 0], R_vel_norm, n_std=2,
            facecolor="none", edgecolor=TEAL, lw=2, ls="-",
            label="Velocity (normalized)")
# Slight offset text to prove overlap is intentional
evals_norm = np.linalg.eigh(R_pos_norm)[0]
lim = np.sqrt(evals_norm.max()) * 2.5
ax.text(0, -lim*0.75, "Both ellipses overlap perfectly\n— same J and $(A^TA)^{-1}$",
        fontsize=9, ha="center", color=NAVY, style="italic",
        bbox=dict(facecolor="lightyellow", edgecolor=NAVY, boxstyle="round,pad=0.3", alpha=0.9))
ax.axhline(0, color="black", lw=0.5, alpha=0.3)
ax.axvline(0, color="black", lw=0.5, alpha=0.3)
ax.set_aspect("equal"); ax.legend(fontsize=9, loc="upper left")
ax.set_xlabel("normalized"); ax.set_ylabel("normalized")
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_title("C: Same shape — both chains\nshare J and $(A^TA)^{-1}$",
             fontweight="bold", fontsize=10)

plt.suptitle("Position and velocity uncertainty: same orientation, different units and scale",
             fontweight="bold", fontsize=12)
plt.tight_layout(); plt.show()

print(f"Position: σ_major={np.sqrt(eigvals_p[1])*1000:.1f}mm, σ_minor={np.sqrt(eigvals_p[0])*1000:.1f}mm")
print(f"Velocity: σ_major={np.sqrt(eigvals_v_plot[1])*1000:.0f}mm/s, σ_minor={np.sqrt(eigvals_v_plot[0])*1000:.0f}mm/s")
print(f"Both ratios: {np.sqrt(eigvals_p[1]/eigvals_p[0]):.1f}:1 — identical because same J and (A^TA)^{{-1}}")


---
## Part 5: Configuration Dependence

In [ ]:
configs = [(np.radians(30),np.radians(90),'Extended'),(np.radians(55),np.radians(75),'Q_REF'),
           (np.radians(80),np.radians(60),'Raised'),(np.radians(45),np.radians(130),'Folded'),
           (np.radians(70),np.radians(40),'Near extended'),(np.radians(30),np.radians(130),'Folded fwd')]
fig, axes = plt.subplots(2, 3, figsize=(16, 11)); axes = axes.flatten()
for idx,(q1,q2,lab) in enumerate(configs):
    ax=axes[idx]; q_cfg=np.array([q1,q2])
    sh,el,hd = draw_arm(ax, q_cfg, lw=5, alpha=0.5)
    J_c=Jfn(q_cfg); R_c=J_c@R_q@J_c.T*100**2; ev,_=np.linalg.eigh(R_c)
    cov_ellipse(ax,hd,R_c,n_std=2,facecolor=PURPLE,alpha=0.15,edgecolor=PURPLE,lw=2,ls='--')
    cov_ellipse(ax,hd,np.eye(2)*0.5**2,n_std=2,facecolor=ORANGE,alpha=0.2,edgecolor=ORANGE,lw=1.5)
    ax.set_aspect('equal'); ax.set_title(f'{lab} ({np.degrees(q1):.0f}\u00b0,{np.degrees(q2):.0f}\u00b0) ratio={np.sqrt(ev[1]/ev[0]):.1f}:1',fontweight='bold',fontsize=10)
    ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
    pad=15; ax.set_xlim(min(sh[0],el[0],hd[0])-pad,max(sh[0],el[0],hd[0])+pad)
    ax.set_ylim(min(sh[1],el[1],hd[1])-pad,max(sh[1],el[1],hd[1])+pad)
plt.suptitle('Proprio (purple) vs peripheral (orange) across configurations',fontsize=13,fontweight='bold',y=1.02)
plt.tight_layout(); plt.show()

---
## Part 7: Position Chain During a Reach — l(t)

Now the position chain evolves during a 10 cm reach. We use **nonlinear FK** at each timestep (not the linearized Jacobian) to avoid artifacts from the linear approximation.

In [ ]:
start_hand = np.array(fk(Q_REF))
tgt = start_hand + np.array([0.10, 0]); T = 0.6
n_steps = int(T/dt)+1; t_arr = np.linspace(0,T,n_steps)
s = 10*(t_arr/T)**3-15*(t_arr/T)**4+6*(t_arr/T)**5
pos_des = start_hand[None,:]+s[:,None]*(tgt-start_hand)[None,:]
q_traj = np.array([ik(p) for p in pos_des])
qd_traj = np.gradient(q_traj, dt, axis=0)
hand_true = np.array([fk(q_traj[i]) for i in range(n_steps)])
ml_true = np.zeros((n_steps,6)); mv_true = np.zeros((n_steps,6))
for i in range(n_steps):
    for j,m in enumerate(muscles):
        ml_true[i,j]=m.length(q_traj[i]); mv_true[i,j]=m.velocity(qd_traj[i])
print(f'Reach: {np.linalg.norm(tgt-start_hand)*100:.0f} cm, T={T*1000:.0f} ms')

In [ ]:
rng_pos = np.random.default_rng(42)
delay_steps = int(0.060/dt)
sample_steps = int(0.010/dt)  # 10ms sampling interval
L_q = np.linalg.cholesky(R_q)

hand_est_pos = np.zeros((n_steps,2))
last_estimate = np.array(fk(Q_REF))  # initial estimate

for i in range(n_steps):
    idx = max(0, i-delay_steps)
    if i % sample_steps == 0:  # new measurement every 10ms
        q_noisy = q_traj[idx] + L_q @ rng_pos.standard_normal(2)
        last_estimate = fk(q_noisy)
    hand_est_pos[i] = last_estimate  # hold between samples

# Identify sample timepoints for plotting
sample_idx = np.arange(0, n_steps, sample_steps)

print(f'Sampling: every {sample_steps}ms, {len(sample_idx)} samples over {T*1000:.0f}ms')
print(f'Mean position error (after delay): {np.mean(np.linalg.norm(hand_est_pos[delay_steps:]-hand_true[delay_steps:],axis=1))*1000:.1f} mm')

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,9)); t_ms=t_arr*1000
names=[m.name for m in muscles]; cm=[TEAL,TEAL,GREEN,ORANGE,ORANGE,RED]

# A: True vs delayed muscle lengths
ax=axes[0,0]
for j in [0,3]:
    ax.plot(t_ms,ml_true[:,j]*100,color=cm[j],lw=2,label=f"{names[j]} (true)")
    dl=np.array([ml_true[max(0,i-delay_steps),j] for i in range(n_steps)])
    ax.plot(t_ms,dl*100,color=cm[j],lw=1,ls="--",alpha=0.5,label=f"{names[j]} (delayed 60ms)")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("Length (cm)")
ax.set_title("A: Muscle lengths — true vs. delayed",fontweight="bold"); ax.legend(fontsize=8)

# B: Hand position — true line + discrete proprio samples
ax=axes[0,1]
ax.plot(t_ms,hand_true[:,0]*100,color=GRAY,lw=2.5,label="True hand x")
valid_samples = sample_idx[sample_idx >= delay_steps]
ax.scatter(t_ms[valid_samples], hand_est_pos[valid_samples,0]*100,
           c=PURPLE, s=20, zorder=5, marker="s",
           label=f"Proprio samples (Δ=60ms, every 10ms)")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("x (cm)")
ax.set_title("B: Hand x-position — true vs. proprioceptive samples",fontweight="bold")
ax.legend(fontsize=9)

# C: Euclidean position error at sample points (discrete, NOT connected)
ax=axes[1,0]
pe_at_samples = np.linalg.norm(hand_est_pos[valid_samples]-hand_true[valid_samples],axis=1)*1000
ax.scatter(t_ms[valid_samples], pe_at_samples, c=PURPLE, s=15, zorder=5,
           label="Euclidean error at each sample")
ax.axhline(12, color="black", ls="--", alpha=0.5, label="Mean σ = 12 mm")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("Euclidean error (mm)")
ax.set_title("C: Position error at each proprioceptive sample\n"
             "(discrete points — each is an independent measurement)",fontweight="bold")
ax.legend(fontsize=9)

# D: Trajectory with ellipses — ZOOMED to hand region, no arm
ax=axes[1,1]
ax.plot(hand_true[:,0]*100,hand_true[:,1]*100,color=GRAY,lw=2.5,label="True trajectory")
ax.scatter(hand_est_pos[valid_samples,0]*100, hand_est_pos[valid_samples,1]*100,
           c=PURPLE, s=12, alpha=0.5, marker="s", label="Proprio samples")
for t_idx in [int(0.15/dt),int(0.30/dt),int(0.45/dt)]:
    Jt=Jfn(q_traj[t_idx]); Rt=Jt@R_q@Jt.T*100**2
    cov_ellipse(ax,hand_true[t_idx]*100,Rt,n_std=2,facecolor=PURPLE,alpha=0.1,edgecolor=PURPLE,lw=1.5,ls="--")
ax.plot(tgt[0]*100,tgt[1]*100,"k+",ms=12,mew=2)
ax.plot(start_hand[0]*100,start_hand[1]*100,"ko",ms=6)
# Zoom to trajectory region
xmin = min(hand_true[:,0].min(), hand_est_pos[valid_samples,0].min())*100 - 1
xmax = max(hand_true[:,0].max(), hand_est_pos[valid_samples,0].max())*100 + 1
ymin = min(hand_true[:,1].min(), hand_est_pos[valid_samples,1].min())*100 - 1.5
ymax = max(hand_true[:,1].max(), hand_est_pos[valid_samples,1].max())*100 + 1.5
ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
ax.set_aspect("equal"); ax.legend(fontsize=8,loc="lower right")
ax.set_xlabel("x (cm)"); ax.set_ylabel("y (cm)")
ax.set_title("D: Reach with ±2σ uncertainty ellipses\n(at 150, 300, 450 ms)",fontweight="bold")

plt.suptitle("Part 7: Position chain l(t) during a 10 cm reach (sampled at 10 ms)",
             fontweight="bold",fontsize=13)
plt.tight_layout(); plt.show()


**Observe:** The estimate lags by 60ms and scatters with anisotropic noise. Error peaks during the fast phase. The delay is the dominant error source, noise is secondary.

---
## Part 8: Velocity Chain During the Same Reach — dl/dt(t)

Ia afferents measure dl/dt **directly** — no differentiation of position needed.
The velocity noise chain follows the same path as position:
σ_v (spindle) → A → R_qdot → J → R_hand_vel.

Here we show the Ia velocity channel during the same 10 cm reach from Part 7.


In [ ]:
rng_vel = np.random.default_rng(99)
L_qd = np.linalg.cholesky(R_qd)

# Ia velocity: delayed + noisy joint velocity mapped to hand velocity
hand_est_vel = np.zeros((n_steps,2))
hand_est_vel_samples = []  # store sample-point velocities
vel_sample_idx = []
for i in range(n_steps):
    idx = max(0, i-delay_steps)
    if i % sample_steps == 0:
        qd_noisy = qd_traj[idx] + L_qd @ rng_vel.standard_normal(2)
        hand_est_vel[i] = Jfn(q_traj[idx]) @ qd_noisy
        if i >= delay_steps:
            hand_est_vel_samples.append(hand_est_vel[i].copy())
            vel_sample_idx.append(i)
    else:
        hand_est_vel[i] = hand_est_vel[max(0,i-1)]  # hold between samples

hand_vel_true = np.gradient(hand_true, dt, axis=0)
vel_sample_idx = np.array(vel_sample_idx)
hand_est_vel_samples = np.array(hand_est_vel_samples)

print(f"Ia velocity noise: σ_v = {sigma_v*1000:.0f} mm/s per muscle")
print(f"Mean velocity error: {np.mean(np.linalg.norm(hand_est_vel[delay_steps:]-hand_vel_true[delay_steps:],axis=1))*1000:.1f} mm/s")


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,9))

# A: Ia velocity signals from muscles (true vs noisy delayed)
ax=axes[0,0]
for j in [0, 3]:
    ax.plot(t_ms,mv_true[:,j]*1000,color=cm[j],lw=2,label=f"{names[j]} (true)")
    dv=np.array([mv_true[max(0,i-delay_steps),j] for i in range(n_steps)])
    ax.plot(t_ms,(dv+np.random.default_rng(j).normal(0,sigma_v,n_steps))*1000,
            color=cm[j],lw=0.5,alpha=0.5,label=f"{names[j]} (noisy Ia, delayed 60ms)")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("mm/s")
ax.set_title("A: Muscle velocities — true vs. noisy Ia signal",fontweight="bold")
ax.legend(fontsize=8)

# B: Hand velocity — Ia direct measurement
ax=axes[0,1]
ax.plot(t_ms,hand_vel_true[:,0]*1000,color=GRAY,lw=2.5,label="True hand vx")
ax.scatter(t_ms[vel_sample_idx], hand_est_vel_samples[:,0]*1000,
           c=PURPLE, s=15, zorder=5, marker="s",
           label="Ia velocity samples (Δ=60ms, every 10ms)")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("vx (mm/s)")
ax.set_title("B: Hand x-velocity — true vs. Ia samples",fontweight="bold")
ax.legend(fontsize=9)

# C: Velocity error at sample points (Euclidean, discrete)
ax=axes[1,0]
ve_at_samples = np.linalg.norm(hand_est_vel_samples - hand_vel_true[vel_sample_idx],axis=1)*1000
ax.scatter(t_ms[vel_sample_idx], ve_at_samples, c=PURPLE, s=15, zorder=5,
           label="Euclidean velocity error at each sample")
mean_vel_noise = np.sqrt(np.mean(np.diag(R_hand_vel)))*1000
ax.axhline(mean_vel_noise, color="black", ls="--", alpha=0.5,
           label=f"Mean σ_vel = {mean_vel_noise:.0f} mm/s")
ax.set_xlabel("Time (ms)"); ax.set_ylabel("Velocity error (mm/s)")
ax.set_title("C: Velocity error at each Ia sample\n"
             "(discrete — each is an independent measurement)",fontweight="bold")
ax.legend(fontsize=9)

# D: Velocity ellipses along the reach — zoomed
ax=axes[1,1]
ax.plot(hand_true[:,0]*100,hand_true[:,1]*100,color=GRAY,lw=2.5,label="Trajectory")
for t_idx in [int(0.15/dt),int(0.30/dt),int(0.45/dt)]:
    Jt=Jfn(q_traj[t_idx]); Rv=Jt@R_qd@Jt.T*1e6  # mm²/s²
    # Scale ellipses for visibility (velocity units don't match position axes)
    # Show as overlay at hand position, scaled down
    Rv_scaled = Rv * 0.0001  # scale to cm² for overlay on position plot
    cov_ellipse(ax,hand_true[t_idx]*100,Rv_scaled,n_std=2,
               facecolor=TEAL,alpha=0.1,edgecolor=TEAL,lw=1.5,ls="--")
ax.plot(tgt[0]*100,tgt[1]*100,"k+",ms=12,mew=2)
ax.plot(start_hand[0]*100,start_hand[1]*100,"ko",ms=6)
xmin = hand_true[:,0].min()*100 - 1; xmax = hand_true[:,0].max()*100 + 1
ymin = hand_true[:,1].min()*100 - 1.5; ymax = hand_true[:,1].max()*100 + 1.5
ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
ax.set_aspect("equal"); ax.legend(fontsize=8)
ax.set_xlabel("x (cm)"); ax.set_ylabel("y (cm)")
ax.set_title("D: Velocity uncertainty ellipses along the reach\n(at 150, 300, 450 ms, scaled for overlay)",
             fontweight="bold")

plt.suptitle("Part 8: Velocity chain dl/dt(t) during the same reach — Ia afferents (direct measurement)",
             fontweight="bold",fontsize=13)
plt.tight_layout(); plt.show()


**Key observation:** The Ia velocity channel (panels A–B) tracks the true hand velocity
with moderate noise (~39 mm/s at the hand). The velocity noise chain follows the same
A and J path as the position chain, producing ellipses with the same 2.3:1 orientation
but in velocity units (mm/s). The Ia measurement is direct — no computation is performed
on the position signal to extract velocity.


---
## Part 9: Both Channels Together During the Reach

Proprioception delivers both position and velocity at every timestep.
Here we plot both channels side by side during the same 10 cm reach,
showing the complete proprioceptive information available to the brain.


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,9))
ax=axes[0,0]
draw_arm(ax,q_traj[0],lw=5,alpha=0.4); draw_arm(ax,q_traj[-1],lw=5,alpha=0.2)
ax.plot(hand_true[:,0]*100,hand_true[:,1]*100,color=GRAY,lw=3,label='True')
ax.plot(hand_est_pos[delay_steps:,0]*100,hand_est_pos[delay_steps:,1]*100,color=PURPLE,lw=1,alpha=0.5,label='Position (group II)')
ax.plot(tgt[0]*100,tgt[1]*100,'k+',ms=14,mew=2.5)
ax.set_aspect('equal'); ax.legend(fontsize=9,loc='lower left'); ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)'); ax.set_title('A: Position channel (group II)',fontweight='bold')

ax=axes[0,1]
draw_arm(ax,q_traj[0],lw=5,alpha=0.4); draw_arm(ax,q_traj[-1],lw=5,alpha=0.2)
ax.plot(hand_true[:,0]*100,hand_true[:,1]*100,color=GRAY,lw=3)
ask=int(0.050/dt); vsc=15
for i in range(delay_steps,n_steps,ask):
    hx,hy=hand_true[i]*100
    vt=hand_vel_true[i]*vsc; ve=hand_est_vel[i]*vsc
    ax.annotate('',xy=(hx+vt[0],hy+vt[1]),xytext=(hx,hy),arrowprops=dict(arrowstyle='->',color=GRAY,lw=1.5,alpha=0.5))
    ax.annotate('',xy=(hx+ve[0],hy+ve[1]),xytext=(hx,hy),arrowprops=dict(arrowstyle='->',color=PURPLE,lw=1.5,alpha=0.7))
ax.plot(tgt[0]*100,tgt[1]*100,'k+',ms=14,mew=2.5)
ax.set_aspect('equal'); ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)'); ax.set_title('B: Velocity channel (group Ia) — true (gray) vs estimate (purple)',fontweight='bold',fontsize=9)

ax=axes[1,0]
ax.plot(t_ms,hand_true[:,0]*100,color=GRAY,lw=2,label='True x')
ax.plot(t_ms,hand_est_pos[:,0]*100,color=PURPLE,lw=1,alpha=0.6,label='Proprio x (group II)')
ax.fill_between(t_ms,(hand_est_pos[:,0]-0.012)*100,(hand_est_pos[:,0]+0.012)*100,color=PURPLE,alpha=0.08)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('x (cm)'); ax.set_title('C: Position state \u00b11\u03c3',fontweight='bold'); ax.legend(fontsize=9)

ax=axes[1,1]
ax.plot(t_ms,hand_vel_true[:,0]*1000,color=GRAY,lw=2,label='True vx')
ax.plot(t_ms,hand_est_vel[:,0]*1000,color=PURPLE,lw=1,alpha=0.6,label='Proprio vx (group Ia)')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('mm/s'); ax.set_title('D: Velocity state (group Ia)',fontweight='bold'); ax.legend(fontsize=9)

plt.suptitle('Part 9: Both proprioceptive channels during the reach\n'
             'The Kalman filter receives the full state [x, y, vx, vy] at every timestep',
             fontweight='bold',fontsize=13)
plt.tight_layout(); plt.show()

**The complete picture:** At every timestep, proprioception delivers
$[\hat{x}, \hat{y}, \hat{v}_x, \hat{v}_y]$ — both position and velocity:
- **Position** (group II) — 60 ms delay, ~12 mm noise (anisotropic, 2.3:1 ratio)
- **Velocity** (group Ia) — 60 ms delay, ~39 mm/s noise (same anisotropy), measured directly

Both channels share the same A and J — same ellipse shape, different scale.
The lecture notes (§4–§5) describe how these signals feed into the Kalman filter
as the proprioceptive observation equation y_proprio = x + ε, with
R_proprio = diag(R_hand_pos, R_hand_vel).


---
## Summary

**Position chain** (group II): spindle length noise (σ_l) → A → R_q → J → R_hand (~12 mm, 2.3:1)

**Velocity chain** (group Ia): spindle velocity noise (σ_v) → same A → R_qdot → same J → R_hand_vel (~39 mm/s, 2.3:1)

**Key takeaways:**
- Both chains use the same A and J — same ellipse shape, different scale
- The 12 mm noise is geometric (Jacobian amplification), not poor spindle acuity
- Noise is **anisotropic** — elongated along the forearm direction
- Ia afferents provide velocity **directly** — no differentiation of position needed
- The ellipse shape and orientation change with arm configuration (Part 5)
- During a reach, both channels track the hand with 60 ms delay (Parts 7–9)
